In [1]:
import numpy as np
np.random.seed(42)

from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_selection import SequentialFeatureSelector

In [2]:
import helper
from indices import *

import tracemalloc
import time

In [3]:
data = [None]*2
data[0] = np.load("dataset_cropped_64_npy\\TRAIN_HEALTHY_even.npy") # healthy
data[1] = np.load("dataset_cropped_64_npy\\TRAIN_STRESSED_even.npy") # stressed

In [4]:
labels = np.concatenate([np.zeros(data[0].shape[1]), np.ones(data[1].shape[1])])
labels.shape

(38978,)

In [5]:
data[1].shape

(12, 19489)

In [6]:
tracemalloc.start()

In [7]:
band_indexes = list(range(1, 12))
encoder = IndicesClassEncoderEq([NORMP], band_indexes)

feature_id = []
features = []
for i in range(encoder.total_length):
    index = encoder.getIndex(i)
    a = index.args
    if a[0] <= a[1]:
        continue

    feature_id.append(i)
    features.append(np.concatenate([index.getValue(data[0]), index.getValue(data[1])]))

features = np.array(features).swapaxes(0, 1)

In [8]:
features.shape

(38978, 55)

In [9]:
time_start = time.time()

dt = DecisionTreeClassifier()
sfs = SequentialFeatureSelector(dt, n_features_to_select=3, direction="backward")

sfs.fit(features, labels)

KeyboardInterrupt: 

In [10]:
time_end = time.time()
print("Time:", time_end - time_start, "sec")
print("MEM usage:", np.array(tracemalloc.get_traced_memory()) / 1024**2, "mb")
tracemalloc.stop()

Time: 3646.421907901764 sec
MEM usage: [70.03693676 70.79945087] mb


In [11]:
mapping = {
    0: "B1",
    1: "B2",
    2: "B3",
    3: "B4",
    4: "B5",
    5: "B6",
    6: "B7",
    7: "B8",
    8: "B8A",
    9: "B9",
    10: "B11",
    11: "B12"
}

selected = np.nonzero(sfs.get_support())[0]
for id in selected:
    index_id = feature_id[id]
    index = encoder.getIndex(index_id)
    name = getIndexName(index, mapping)
    print("Id:", index_id, "Name:", name)

AttributeError: 'SequentialFeatureSelector' object has no attribute 'support_'